# Bayesian Hyperparameter Tuning

Standalone notebook. Run this **independently** of the pipeline benchmarks to tune
model hyperparameters and persist the best configuration to `runs/hyperparams.json`.

The pipeline benchmark (`pipeline_benchmark.ipynb`) and main demo
(`microstructure_demo.ipynb`) will automatically load and use the saved params on
their next run — no manual copy-paste required.

## Workflow

```
bayes_tuning.ipynb          pipeline_benchmark.ipynb / microstructure_demo.ipynb
──────────────────          ──────────────────────────────────────────────────────
§1 configure scope    →     (reads scope from TUNING_SCOPE constant)
§2 build features
§3 run Optuna (100 trials)
§4 evaluate & save    →     runs/hyperparams.json   ←   §4/§6 load at startup
§5 visualise
```

**Scopes**

| Scope string | Dataset filter | Used by |
|---|---|---|
| `dp_steel` | DP/dual-phase rows only | `pipeline_benchmark.ipynb` |
| `all_alloys` | Full dataset | `microstructure_demo.ipynb` |

In [1]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.insert(0, os.path.abspath('..'))

from src.column_sanitizer import sanitize_dataframe
from src.config import PreprocessingConfig, MissingDataConfig, ScalingConfig, EncodingConfig
from src.preprocessing import FeaturePreprocessor
from src import hyperparams as hp
from src.features import FeaturePipeline

from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               AdaBoostRegressor, ExtraTreesRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, make_scorer


# Image & morphological feature support
SEED     = 42
multi_r2 = make_scorer(r2_score, multioutput='uniform_average')
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print('Setup complete.')
print()
print('Existing store contents:')
print(hp.summary())
from src.run_store import RunStore
from src import metrics_viz


Setup complete.

Existing store contents:
Scope: dp_steel
  saved_at:   2026-04-20 05:55:57
  git_commit: 059e806
  n_trials:   100
  cv:         RepeatedKFold(n_splits=5, n_repeats=5) search / (n_repeats=10) eval
  best_model: GBR
    GBR            tuned_cv_r2=0.8836378593581563  delta=0.08319043089994993 ◀
    ExtraTrees     tuned_cv_r2=0.7389852052897882  delta=0.016912118261879105
    ABR            tuned_cv_r2=0.7719350380374075  delta=0.10830765647350848


## §1 — Configuration

In [ ]:
# ── Choose scope ──────────────────────────────────────────────────────────────
# "dp_steel"   → DP / dual-phase rows only  (used by pipeline_benchmark.ipynb)
# "all_alloys" → full dataset               (used by microstructure_demo.ipynb)
TUNING_SCOPE = "dp_steel"

# ── Tuning budget ─────────────────────────────────────────────────────────────
N_TRIALS = 100    # Optuna trials per model
TOP_N    = 3      # how many top models to tune
N_PARALLEL_TRIALS = 8  # Optuna parallelism (use ~80% of cores)

# ── CV protocols ──────────────────────────────────────────────────────────────
CV_SEARCH = RepeatedKFold(n_splits=5, n_repeats=5,  random_state=SEED)  # fast
CV_EVAL   = RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED)  # final

# ── Preprocessing config (must match what the pipeline uses) ──────────────────
DROP_THRESHOLD = 0.95
IMPUTE         = 'mice'
SCALE          = 'standard'
ENCODE         = 'onehot'

# ── Feature streams ──────────────────────────────────────────────────────────
# Set BACKBONE to a cached backbone name (e.g. "dinov2_vitb14", "resnet50"),
# or None to tune on tabular features only.
BACKBONE = "dinov2_vitb14"

fp = FeaturePipeline(
    data_dir=os.path.abspath('../data'),
    temp_dir=os.path.abspath('../data/temp_images'),
    features_dir=os.path.abspath('../features'),
)
print(f'Scope:          {TUNING_SCOPE}')
print(f'Trials/model:   {N_TRIALS}')
print(f'Top-N to tune:  {TOP_N}')
print(f'Preprocessing:  drop={DROP_THRESHOLD}  impute={IMPUTE}  scale={SCALE}  encode={ENCODE}')

_store = RunStore("bayes")
_run_dir, _run_id = _store.start()
print(f"Run directory: {_run_dir}")

Scope:          all_alloys
Trials/model:   100
Top-N to tune:  3
Preprocessing:  drop=0.95  impute=mice  scale=standard  encode=onehot
Run directory: /Users/ayobamibamigboye/capstone/model/runs/bayes/20260426_105144_65af


## §2 — Load Data & Build Feature Matrix

In [3]:
# ── Load & sanitize ───────────────────────────────────────────────────────────
df_raw = pd.read_csv('../datasets/metadata_latest.csv', header=1)
df_raw = sanitize_dataframe(df_raw)
df_raw = df_raw[df_raw['c'].notna()].copy().reset_index(drop=True)

# ── Dataset filter ────────────────────────────────────────────────────────────
if TUNING_SCOPE == 'dp_steel':
    mask = df_raw['alloy'].str.lower().str.contains(r'dp|dual.?phase', na=False, regex=True)
    df   = df_raw[mask].copy().reset_index(drop=True)
else:
    df   = df_raw.copy()

C1_TEMP    = 'cycle1_holdingtemp_degc'
C1_TIME    = 'cycle1_holdingtime_min'
TARGET_COLS = [C1_TEMP, C1_TIME]

df = df[df[C1_TEMP].notna() & df[C1_TIME].notna()].copy().reset_index(drop=True)

Y = df[TARGET_COLS].values.astype(np.float64)
print(f'Scope={TUNING_SCOPE}: {len(df)} rows, Y shape={Y.shape}')

# ── Chemistry-only allowlist (leakage-free) ───────────────────────────────────
# Mirrors the post-fix morph_ablation pipeline: 14 raw chemistry columns +
# missingness indicators for trace microalloying elements (Ti/Nb/V).
# Excludes any heat-treatment recipe columns (num_cycles, cycle*_qtemp/rtemp,
# cycle2_holdingtemp, etc.) which are co-determined with the targets.
CHEMICAL_COLUMNS = [
    'c', 'mn', 'si', 'cr', 'p', 's',
    'mo', 'cu', 'ni', 'al', 'nb', 'v', 'ti', 'fe',
]
FEATURE_COLS = [c for c in CHEMICAL_COLUMNS if c in df.columns]
print(f'Tabular features (chemistry only, {len(FEATURE_COLS)}): {FEATURE_COLS}')

MICE_COLS      = [c for c in ['cr', 'mo', 's', 'ni', 'al'] if c in FEATURE_COLS]
INDICATOR_COLS = [c for c in ['ti', 'nb', 'v']             if c in FEATURE_COLS]

# ── Build preprocessor ────────────────────────────────────────────────────────
cfg = PreprocessingConfig(
    missing_data=MissingDataConfig(
        column_drop_threshold=DROP_THRESHOLD,
        row_fill_threshold=0.50,
        numeric_fill_strategy=IMPUTE if IMPUTE != 'mice' else 'median',
        categorical_fill_strategy='mode',
        mice_max_iter=10,
    ),
    scaling=ScalingConfig(method=SCALE, enabled=(SCALE != 'none')),
    encoding=EncodingConfig(categorical=ENCODE, max_categories=50),
)
prep = FeaturePreprocessor(cfg,
                           mice_columns=MICE_COLS if IMPUTE == 'mice' else [],
                           indicator_columns=INDICATOR_COLS)
X = prep.fit_transform(df[FEATURE_COLS].copy())
print(f'Tabular feature dim: {X.shape[1]}')

# ── §2b — Image & morphological feature streams ──────────────────────────────
X_img   = fp.load_image_features(BACKBONE, df['id']) if BACKBONE else None

# Morph features — load raw, then median-impute + StandardScale to mirror morph_ablation
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
_X_morph_raw = fp.load_morph_features(df['id'])
if _X_morph_raw is not None:
    _morph_imputer = SimpleImputer(strategy='median')
    _morph_scaler  = StandardScaler()
    X_morph = _morph_scaler.fit_transform(_morph_imputer.fit_transform(_X_morph_raw))
else:
    X_morph = None

_parts = [p for p in [X_img, X_morph] if p is not None] + [X]
X_full = np.concatenate(_parts, axis=1).astype(np.float64)
_stream_desc = ' + '.join(
    ([f'{X_img.shape[1]} (image)']   if X_img   is not None else []) +
    ([f'{X_morph.shape[1]} (morph)'] if X_morph is not None else []) +
    [f'{X.shape[1]} (tabular)']
)
print(f'Combined feature matrix: {X_full.shape}  [{_stream_desc}]')

Scope=all_alloys: 111 rows, Y shape=(111, 2)
Tabular features (chemistry only, 14): ['c', 'mn', 'si', 'cr', 'p', 's', 'mo', 'cu', 'ni', 'al', 'nb', 'v', 'ti', 'fe']
  ti: added indicator 'ti_present', zero-filled
  nb: added indicator 'nb_present', zero-filled
  v: added indicator 'v_present', zero-filled
  MICE fitted on ['cr', 'mo', 's', 'ni', 'al']
  c: float64 → numeric
  mn: float64 → numeric
  si: float64 → numeric
  cr: float64 → numeric
  p: float64 → numeric
  s: float64 → numeric
  mo: float64 → numeric
  cu: float64 → numeric
  ni: float64 → numeric
  al: float64 → numeric
  nb: float64 → numeric
  v: float64 → numeric
  ti: float64 → numeric
  fe: float64 → numeric
  ti_present: float64 → numeric
  nb_present: float64 → numeric
  v_present: float64 → numeric
  Fitted 17 columns, dropped 0, output features: 17
Tabular feature dim: 17
Combined feature matrix: (111, 818)  [768 (image) + 33 (morph) + 17 (tabular)]


## §3 — Quick Regressor Sweep

Runs a 5×5 CV sweep to identify the top-N candidates before tuning.

In [4]:
def multi_output(est):
    return MultiOutputRegressor(est, n_jobs=-1)

CANDIDATES = {
    'RF':         RandomForestRegressor(n_estimators=200, max_features='sqrt',
                                         min_samples_leaf=3, random_state=SEED, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_features='sqrt',
                                       min_samples_leaf=3, random_state=SEED, n_jobs=-1),
    'GBR':        multi_output(GradientBoostingRegressor(
                      n_estimators=200, learning_rate=0.05, max_depth=3,
                      subsample=0.8, min_samples_leaf=5, random_state=SEED)),
    'ABR':        multi_output(AdaBoostRegressor(
                      estimator=DecisionTreeRegressor(max_depth=3),
                      n_estimators=100, random_state=SEED)),
    'SVR_rbf':    multi_output(SVR(kernel='rbf',  C=10, gamma='scale', epsilon=0.1)),
    'SVR_linear': multi_output(SVR(kernel='linear', C=1, epsilon=0.1)),
    'KNN':        multi_output(KNeighborsRegressor(n_neighbors=5)),
}

sweep_results = []
print(f'Sweeping {len(CANDIDATES)} candidates (RepeatedKFold 5x5)...')
for name, model in CANDIDATES.items():
    t0     = time.time()
    scores = cross_val_score(model, X_full, Y, cv=CV_SEARCH, scoring=multi_r2, n_jobs=-1)
    sweep_results.append({'model': name, 'mean_r2': scores.mean(), 'std_r2': scores.std()})
    print(f'  {name:<12} R²={scores.mean():.4f}±{scores.std():.4f}  ({time.time()-t0:.1f}s)')

df_sweep = (pd.DataFrame(sweep_results)
              .sort_values('mean_r2', ascending=False)
              .reset_index(drop=True))
print()
print(df_sweep.to_string(index=False))
top_models = df_sweep.head(TOP_N)['model'].tolist()
print(f'\nTop-{TOP_N} selected for Bayesian tuning: {top_models}')

Sweeping 7 candidates (RepeatedKFold 5x5)...
  RF           R²=0.4512±0.1361  (4.6s)
  ExtraTrees   R²=0.4219±0.1388  (0.5s)
  GBR          R²=0.8148±0.1705  (8.0s)
  ABR          R²=0.8149±0.1512  (3.4s)
  SVR_rbf      R²=0.1277±0.1143  (0.1s)
  SVR_linear   R²=0.2519±0.3135  (0.1s)
  KNN          R²=0.0224±0.4925  (0.1s)

     model  mean_r2   std_r2
       ABR 0.814935 0.151221
       GBR 0.814831 0.170536
        RF 0.451227 0.136137
ExtraTrees 0.421918 0.138849
SVR_linear 0.251896 0.313520
   SVR_rbf 0.127730 0.114311
       KNN 0.022412 0.492537

Top-3 selected for Bayesian tuning: ['ABR', 'GBR', 'RF']


## §4 — Bayesian Optimisation

In [ ]:
# ── Search space definitions ──────────────────────────────────────────────────
def build_model(trial, name):
    if name in ('RF', 'ExtraTrees'):
        p = dict(
            n_estimators    = trial.suggest_int('n_estimators', 50, 600),
            max_depth       = trial.suggest_int('max_depth', 3, 30),
            min_samples_leaf= trial.suggest_int('min_samples_leaf', 1, 10),
            max_features    = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5]),
            bootstrap       = trial.suggest_categorical('bootstrap', [True, False]),
        )
        cls = ExtraTreesRegressor if name == 'ExtraTrees' else RandomForestRegressor
        return cls(**p, random_state=SEED, n_jobs=2)

    elif name == 'GBR':
        p = dict(
            n_estimators    = trial.suggest_int('n_estimators', 50, 500),
            learning_rate   = trial.suggest_float('learning_rate', 1e-3, 0.5, log=True),
            max_depth       = trial.suggest_int('max_depth', 2, 8),
            subsample       = trial.suggest_float('subsample', 0.4, 1.0),
            min_samples_leaf= trial.suggest_int('min_samples_leaf', 1, 20),
            max_features    = trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        )
        return multi_output(GradientBoostingRegressor(**p, random_state=SEED))

    elif name == 'ABR':
        md = trial.suggest_int('max_depth', 1, 8)
        p  = dict(n_estimators = trial.suggest_int('n_estimators', 50, 400),
                  learning_rate= trial.suggest_float('learning_rate', 0.01, 2.0, log=True))
        return multi_output(AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=md), **p, random_state=SEED))

    elif 'SVR' in name:
        kernel = 'linear' if 'linear' in name else 'rbf'
        p = dict(C      = trial.suggest_float('C', 0.01, 1000.0, log=True),
                 epsilon= trial.suggest_float('epsilon', 1e-3, 10.0, log=True))
        if kernel == 'rbf':
            p['gamma'] = trial.suggest_categorical('gamma', ['scale', 'auto'])
        return multi_output(SVR(kernel=kernel, **p))

    elif 'KNN' in name:
        p = dict(n_neighbors= trial.suggest_int('n_neighbors', 2, min(15, len(df) // 5)),
                 weights    = trial.suggest_categorical('weights', ['uniform', 'distance']),
                 p          = trial.suggest_int('p', 1, 2))
        return multi_output(KNeighborsRegressor(**p))

    raise ValueError(f'No search space for: {name}')


class _FixedTrial:
    def __init__(self, p):                       self._p = p
    def suggest_int(self, n, *a, **kw):          return self._p[n]
    def suggest_float(self, n, *a, **kw):        return self._p[n]
    def suggest_categorical(self, n, *a, **kw):  return self._p[n]


# ── Run Optuna ────────────────────────────────────────────────────────────────
studies = {}
for model_name in top_models:
    print(f'Optimising {model_name} ({N_TRIALS} trials)...')
    t0 = time.time()

    def _obj(trial, _n=model_name):
        return float(cross_val_score(
            build_model(trial, _n), X_full, Y,
            cv=CV_SEARCH, scoring=multi_r2, n_jobs=1).mean())

    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=SEED, n_startup_trials=20),
        pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5),
    )

    # ── Progress callback ──────────────────────────────────────────────────
    # Optuna fires this after every completed trial.  Works with n_jobs>1
    # because Optuna serialises callback execution.  We print every trial
    # once the search is past the warm-up phase, otherwise every 10 trials.
    _progress_state = {'last_print': 0, 't_start': time.time()}
    def _on_trial(study_, trial):
        n_done = len(study_.trials)
        # Throttle: print every trial in the first 20 (warm-up), then every 5
        if n_done <= 20 or (n_done - _progress_state['last_print']) >= 5 or n_done == N_TRIALS:
            _progress_state['last_print'] = n_done
            elapsed = time.time() - _progress_state['t_start']
            best = study_.best_value if study_.best_trial else float('nan')
            val  = trial.value if trial.value is not None else float('nan')
            print(f'    [{model_name}] trial {n_done:3d}/{N_TRIALS}  '
                  f'this_R²={val:+.4f}  best_so_far={best:.4f}  '
                  f'({elapsed:5.1f}s elapsed)')

    study.optimize(_obj, n_trials=N_TRIALS, n_jobs=N_PARALLEL_TRIALS,
                   show_progress_bar=False, callbacks=[_on_trial])
    studies[model_name] = study

    # ── Persist per-trial trajectory ────────────────────────────────────────
    # Captures every trial's R² so convergence plots use real data, not a
    # stylised reconstruction.  best_so_far is the running maximum.
    _trials_df = study.trials_dataframe(attrs=('number', 'value', 'state', 'datetime_start', 'datetime_complete'))
    _trials_df = _trials_df.rename(columns={'value': 'cv_r2'})
    _trials_df['best_so_far'] = _trials_df['cv_r2'].cummax()
    _trials_csv = _run_dir / f'optuna_trials_{model_name}.csv'
    _trials_df.to_csv(_trials_csv, index=False)
    print(f'  best R²={study.best_value:.4f}  ({time.time()-t0:.0f}s)  '
          f'trajectory saved → {_trials_csv.name}')
    print(f'    params={study.best_params}')
    print()

# ── Final evaluation — RepeatedKFold(5x10) ───────────────────────────────────
print('Final eval RepeatedKFold(5x10):')
print('=' * 55)
models_out = {}
for model_name in top_models:
    study  = studies[model_name]
    model  = build_model(_FixedTrial(study.best_params), model_name)
    sc     = cross_val_score(model, X_full, Y, cv=CV_EVAL, scoring=multi_r2, n_jobs=-1)
    base   = df_sweep.loc[df_sweep['model'] == model_name, 'mean_r2'].values[0]
    delta  = sc.mean() - base
    models_out[model_name] = {
        'best_cv_r2':  study.best_value,
        'tuned_cv_r2': float(sc.mean()),
        'tuned_std':   float(sc.std()),
        'delta':       float(delta),
        'params':      study.best_params,
    }
    marker = ' \u2b50' if delta > 0 else ''
    print(f'  {model_name:<12} untuned={base:.4f}  '
          f'tuned={sc.mean():.4f}\u00b1{sc.std():.4f}  \u0394={delta:+.4f}{marker}')

best_name = max(models_out, key=lambda k: models_out[k]['tuned_cv_r2'])
print(f'\nBest tuned model: {best_name}  CV R\u00b2={models_out[best_name]["tuned_cv_r2"]:.4f}')

# ── Save to hyperparameter store ──────────────────────────────────────────────
store_path = hp.save(
    scope=TUNING_SCOPE,
    models_dict=models_out,
    n_trials=N_TRIALS,
    cv_protocol=f'RepeatedKFold(n_splits=5, n_repeats=5) search / (n_repeats=10) eval',
    feature_matrix_shape=X_full.shape,
)
print(f'\nSaved to: {store_path}')
print()
print(hp.summary())

# ── Write manifest ────────────────────────────────────────────────────────────
_store.write_manifest({
    'scope':               TUNING_SCOPE,
    'n_trials':            N_TRIALS,
    'feature_matrix_rows': int(X_full.shape[0]),
    'feature_matrix_cols': int(X_full.shape[1]),
    'top_n':               TOP_N,
    'cv_search':           'RepeatedKFold(5,5)',
    'cv_eval':             'RepeatedKFold(5,10)',
    'best_model':          best_name,
    'models':              models_out,
    'trials_csvs':         {n: f'optuna_trials_{n}.csv' for n in top_models},
})

# ── Append one row per tuned model to history ─────────────────────────────────
_hist_row = {
    'scope':               TUNING_SCOPE,
    'n_trials':            N_TRIALS,
    'feature_matrix_rows': int(X_full.shape[0]),
    'feature_matrix_cols': int(X_full.shape[1]),
    'best_model':          best_name,
}
for _mn, _mv in models_out.items():
    _hist_row[f'{_mn}_tuned_cv_r2'] = float(_mv.get('tuned_cv_r2', float('nan')))
    _hist_row[f'{_mn}_delta']        = float(_mv.get('delta', float('nan')))
_store.append_history(_hist_row)


Optimising ABR (100 trials)...


[W 2026-04-26 10:53:08,477] Trial 9 failed with parameters: {'max_depth': 2, 'n_estimators': 244, 'learning_rate': 0.02063738528749071} because of the following error: KeyboardInterrupt().
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/Users/ayobamibamigboye/capstone/model/venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py", line 490, in _process_worker
    r = call_item()
  File "/Users/ayobamibamigboye/capstone/model/venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py", line 291, in __call__
    return self.fn(*self.args, **self.kwargs)
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ayobamibamigboye/capstone/model/venv/lib/python3.13/site-packages/joblib/parallel.py", line 607, in __call__
    return [func(*args, **kwargs) for func, args, kwargs in self.items]
            ~~~~^^^^^^^^^^^^^^^^^
  File "/Users/ayobamibamigboye/capstone/model/venv/lib/python3.13/site

## §5 — Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — untuned vs tuned
ax     = axes[0]
names  = list(models_out.keys())
x      = np.arange(len(names))
w      = 0.35
base_r2 = [df_sweep.loc[df_sweep['model']==n, 'mean_r2'].values[0] for n in names]
tuned_r2 = [models_out[n]['tuned_cv_r2'] for n in names]
tuned_std= [models_out[n]['tuned_std']   for n in names]

ax.bar(x - w/2, base_r2,  w, label='Untuned (sweep)', alpha=0.7, color='steelblue')
ax.bar(x + w/2, tuned_r2, w, label='Bayesian-tuned',  alpha=0.85, color='#2ecc71',
       yerr=tuned_std, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Mean CV R\u00b2')
ax.set_title(f'Tuning Impact — {TUNING_SCOPE}', fontweight='bold')
ax.legend()
for xi, (u, t, e) in enumerate(zip(base_r2, tuned_r2, tuned_std)):
    col = '#27ae60' if t >= u else '#e74c3c'
    ax.text(xi + w/2, t + e + 0.005, f'{t-u:+.3f}',
            ha='center', va='bottom', fontsize=9, color=col, fontweight='bold')

# Right — optimisation history
ax2  = axes[1]
cmap = plt.cm.tab10.colors
for ci, model_name in enumerate(top_models):
    study = studies[model_name]
    vals  = [t.value for t in study.trials if t.value is not None]
    ax2.plot(range(1, len(vals)+1), np.maximum.accumulate(vals),
             label=model_name, color=cmap[ci], linewidth=1.8)
ax2.set_xlabel('Trial')
ax2.set_ylabel('Best CV R\u00b2 so far')
ax2.set_title('Optuna Optimisation History', fontweight='bold')
ax2.legend()

plt.suptitle(f'Bayesian Hyperparameter Optimisation — {TUNING_SCOPE}',
             fontsize=13, fontweight='bold')
plt.tight_layout()

out_png = str(_run_dir / f'bayes_tuning_{TUNING_SCOPE}.png')
plt.savefig(out_png, dpi=150)
plt.show()
print(f'Saved {out_png}')

# ── FAnova parameter importance ───────────────────────────────────────────────
print('\nParameter importance (FAnova):')
print('\u2500' * 55)
for model_name in top_models:
    try:
        imp = optuna.importance.get_param_importances(studies[model_name])
        print(f'\n{model_name}:')
        for k, v in imp.items():
            print(f'  {k:<24} {v:.3f}  {"\u2588" * int(v * 30)}')
    except Exception as e:
        print(f'  {model_name}: {e}')
# ── Regenerate history visualisation ─────────────────────────────────────────
metrics_viz.plot_history('bayes', save=True, show=False)
print(f'History visualisation -> runs/bayes/metrics_history.png')
print(f'Run artifacts stored  -> {_run_dir}')


## §6 — Verify the Saved Params

Run this cell at any time to inspect what is currently stored — no tuning required.

In [ ]:
import json as _json

print(hp.summary())
print()

# Show the winning params in full
name, params = hp.best_model(TUNING_SCOPE)
if name:
    print(f'Best model for scope=\'{TUNING_SCOPE}\': {name}')
    print(_json.dumps(params, indent=2))
else:
    print(f'No params saved for scope=\'{TUNING_SCOPE}\' yet.')